In [2]:
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    PredefinedSplit,
    train_test_split,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

wine = pd.read_csv("winequality-white.csv", sep=None, engine="python")
X = wine.drop(columns="quality")
y = wine["quality"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.125, random_state=42
)

X_search = pd.concat([X_train, X_val], ignore_index=True)
y_search = pd.concat([y_train, y_val], ignore_index=True)
validation_split = PredefinedSplit([-1] * len(X_train) + [0] * len(X_val))

knn = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsRegressor()),
])
knn_grid = {
    "knn__n_neighbors": range(1, 91),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan", "chebyshev"],
}

forest = RandomForestRegressor(random_state=42)
forest_grid = {
    "n_estimators": [12, 24, 36, 48, 60],
    "max_depth": [4, 8, 12, 16, None],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [0.3, 0.45, 0.6, 0.8, 1.0],
}

for name, model, grid in [
    ("KNN", knn, knn_grid),
    ("Random forest", forest, forest_grid),
]:
    n_combinations = len(ParameterGrid(grid))
    assert n_combinations > len(X_val)

    search = GridSearchCV(
        model,
        grid,
        cv=validation_split,
        scoring="neg_mean_squared_error",
        refit=False,
        n_jobs=4,
        error_score="raise",
    )
    search.fit(X_search, y_search)

    best_model = clone(model).set_params(**search.best_params_)
    best_model.fit(X_train, y_train)

    val_mse = -search.best_score_
    test_mse = mean_squared_error(y_test, best_model.predict(X_test))

    print(f"{name} | Parameter combinations: {n_combinations}")
    print(f"Validation MSE: {val_mse:.6f}")
    print(f"Test MSE: {test_mse:.6f}")

KNN | Parameter combinations: 540
Validation MSE: 0.298739
Test MSE: 0.368550
Random forest | Parameter combinations: 500
Validation MSE: 0.308138
Test MSE: 0.355338
